# Generating MoTrPAC GMTs via DGE with DESeq2

This is the second step in the workflow
While the processed rnaseq files are sufficient to submit individually to the chea-kg-ts site, to reduce runtime of the example reports we will precompute the dges here

In [7]:
# general
import pandas as pd
import numpy as np
import json
import requests
import urllib
from IPython.display import display, FileLink, HTML, Markdown
import time
import os
import glob

# differential gene expression
from maayanlab_bioinformatics.dge import deseq2_differential_expression, characteristic_direction, up_down_from_characteristic_direction

import sys
import contextlib
@contextlib.contextmanager
def suppress_output(stdout=True, stderr=True, dest=os.devnull):
    ''' Usage:
    with suppress_output():
        print('hi')
    '''
    dev_null = open(dest, 'a')
    if stdout:
        _stdout = sys.stdout
        sys.stdout = dev_null
    if stderr:
        _stderr = sys.stderr
        sys.stderr = dev_null
    try:
        yield
    finally:
        if stdout:
            sys.stdout = _stdout
        if stderr:
            sys.stderr = _stderr

In [ ]:
def get_gene_lists(input_csv):
    """
    Outputs list of upregulated DEGs from DESeq2 results CSV.
    """
    df = pd.read_csv(input_csv)

    ## up genes ##
    filtered = df.loc[df["log2FoldChange"] > 0, df.columns[0]]
    up_gene_ids = list(filtered)

    up_list = []
    for up_gene in up_gene_ids:
        if "_" in up_gene:
            up_list.append(up_gene.split("_", 1)[1])
        else:
            up_list.append(up_gene)

    ## down genes ##
    filtered = df.loc[df["log2FoldChange"] < 0, df.columns[0]]
    dn_gene_ids = list(filtered)

    down_list = []
    for dn_gene in dn_gene_ids:
        if "_" in dn_gene:
            down_list.append(dn_gene.split("_", 1)[1])
        else:
            down_list.append(dn_gene)

    return up_list, down_list


def deseq2_csv_to_gmt(input_csv_list, comparisons, filename):
    """
    Converts list of csv files to two GMTs, comparing time vs 0 and time vs time
    """
    gmt_dict = {}
    trimmed_filenames = {x: x.replace("out/rat/dge/", "").replace("_deseq2","").split(".")[0].replace("_"," ") for x in input_csv_list}
    results_table = pd.DataFrame(index=trimmed_filenames.values(), columns=["up genes", "down genes"])

    for i, file in enumerate(input_csv_list):
        up_genes, down_genes = get_gene_lists(file)
        results_table.loc[trimmed_filenames[file], "up genes"] = len(up_genes)
        results_table.loc[trimmed_filenames[file], "down genes"] = len(down_genes)
        gmt_dict[f"{comparisons[i]} up genes"] = up_genes
        gmt_dict[f"{comparisons[i]} down genes"] = down_genes

    with open(filename, "w") as file:
        for s,t in gmt_dict.items():
            file.write(str(s) + "\t\t" + "\t".join(t) + "\n")

    results_table.index.name = "comparison"
    
    return filename, results_table

In [ ]:
def chea_kg_ts_dge(prefix, count_matrix, metadata):
    raw_counts = pd.read_csv(count_matrix)
    sample_metadata = pd.read_csv(metadata)

    assert sample_metadata['time_pt_annotation'].value_counts().min() >= 2, "Each time point must have at least two replicates"

    time_pt_list = [] 

    for time_pt in sample_metadata["time_pt_annotation"].tolist():
        if time_pt not in time_pt_list:
            time_pt_list.append(time_pt) 
    
    time_pt_dict = {} 
    for i, time_pt in enumerate(time_pt_list):
        samples_at_time_pt = sample_metadata.loc[sample_metadata["time_pt_annotation"] == time_pt, "sample_name"].astype(str).tolist()
        if raw_counts.columns[0] == "gene_id": # TODO: set raw count index to genes
            subset_counts = raw_counts[["gene_id"] + samples_at_time_pt]
            rev_subset_counts = subset_counts.set_index("gene_id")
            time_pt_dict[i] = (time_pt, rev_subset_counts)
        elif raw_counts.columns[0] == "gene_name":
            subset_counts = raw_counts[["gene_name"] + samples_at_time_pt]
            rev_subset_counts = subset_counts.set_index("gene_name")
            time_pt_dict[i] = (time_pt, rev_subset_counts)
    
    adj_time_pt_comparisons = []
    adj_time_pt_degs = []

    for i in range(len(time_pt_list) - 1):
        controls, cases = time_pt_dict[i][1], time_pt_dict[i+1][1]
        with suppress_output():
            results_df = deseq2_differential_expression(controls, cases)

        p_vals = [0.05] + [10**(-i) for i in range(2, 21, 1)]
        significant_genes = results_df[results_df["padj"] < p_vals[0]]
        up_count = (significant_genes["log2FoldChange"] > 0).sum()
        down_count = (significant_genes["log2FoldChange"] < 0).sum()

        idx = 1
        while (up_count + down_count) > 2000:
            significant_genes = results_df[results_df["padj"] < p_vals[idx]]
            up_count = (significant_genes["log2FoldChange"] > 0).sum()
            down_count = (significant_genes["log2FoldChange"] < 0).sum()
            idx += 1

        ctrl_time_pt, case_time_pt = time_pt_dict[i][0], time_pt_dict[i+1][0]
        adj_time_pt_comparisons.append(f"{ctrl_time_pt} v {case_time_pt}")

        file = f"out/rat/dge/{prefix}_deseq2_{case_time_pt}_v_{ctrl_time_pt}.csv"
        significant_genes.to_csv(file)
        adj_time_pt_degs.append(file)
    
    time_pt_0_comparisons = []
    time_pt_0_degs = [] 
    for i in range(1, len(time_pt_list)):
        controls, cases = time_pt_dict[0][1], time_pt_dict[i][1]
        with suppress_output():
            results_df = deseq2_differential_expression(controls, cases)

        p_vals = [0.05] + [10**(-i) for i in range(2, 21, 1)]
        significant_genes = results_df[results_df["padj"] < p_vals[0]]
        up_count = (significant_genes["log2FoldChange"] > 0).sum()
        down_count = (significant_genes["log2FoldChange"] < 0).sum() 

        idx = 1
        while (up_count + down_count) > 2000:
            significant_genes = results_df[results_df["padj"] < p_vals[idx]]
            up_count = (significant_genes["log2FoldChange"] > 0).sum()
            down_count = (significant_genes["log2FoldChange"] < 0).sum()
            idx += 1

        ctrl_time_pt, case_time_pt = time_pt_dict[0][0], time_pt_dict[i][0]
        time_pt_0_comparisons.append(f"{ctrl_time_pt} v {case_time_pt}")

        file = f"out/rat/dge/{prefix}_{case_time_pt}_v_{ctrl_time_pt}.csv"
        significant_genes.to_csv(file)
        time_pt_0_degs.append(file)
    
    display(Markdown("**Adjacent Time Point Comparisons**"))
    deseq2_degs_gmt_1, results_table1 = deseq2_csv_to_gmt(adj_time_pt_degs, adj_time_pt_comparisons, f"out/rat/gmts/motrpac_rat_{prefix}_adj_time_pt_degs.gmt")
    for file in adj_time_pt_degs:
        trimmed_filename = file.replace("results/deseq2_", "").split(".")[0].replace("_"," ")
    display(results_table1)

    display(Markdown("**Time Point Versus 0 Comparisons**"))
    deseq2_degs_gmt_2, results_table2 = deseq2_csv_to_gmt(time_pt_0_degs, time_pt_0_comparisons, f"out/rat/gmts/motrpac_rat_{prefix}_time_pt_0_degs.gmt")  
    for file in time_pt_0_degs:
        trimmed_filename = file.replace("results/deseq2_", "").split(".")[0].replace("_"," ")
    display(results_table2) 


In [ ]:
all_results = dict()
for tissue in os.listdir("./out/rat/processed_rnaseq"):
    counts_matrix = f"./out/rat/processed_rnaseq/{tissue}/{tissue}_rna-seq_named.txt"
    
    M_prefix = f"{tissue}_male"
    male_metadata = f"./out/rat/processed_rnaseq/{tissue}/{tissue}_male_samples.csv"
    results = chea_kg_ts_dge(M_prefix, counts_matrix, male_metadata)
    all_results[M_prefix] = results

    F_prefix = f"{tissue}_female"
    female_metadata = f"./out/rat/processed_rnaseq/{tissue}/{tissue}_female_samples.csv"
    results = chea_kg_ts_dge(F_prefix, counts_matrix, female_metadata)
    all_results[F_prefix] = results

**Adjacent Time Point Comparisons**

,up genes,down genes
comparison,,
blood-rna male male 1w v male sedentary,465,131
blood-rna male male 2w v male 1w,0,0
blood-rna male male 4w v male 2w,214,393
blood-rna male male 8w v male 4w,1024,361


**Time Point Versus 0 Comparisons**

,up genes,down genes
comparison,,
blood-rna male male 1w v male sedentary,465,131
blood-rna male male 2w v male sedentary,697,462
blood-rna male male 4w v male sedentary,0,0
blood-rna male male 8w v male sedentary,1001,284


**Adjacent Time Point Comparisons**

...

,up genes,down genes
comparison,,
gastrocnemius male male 1w v male sedentary,127,68
gastrocnemius male male 2w v male sedentary,55,53
gastrocnemius male male 4w v male sedentary,45,37
gastrocnemius male male 8w v male sedentary,239,294


**Adjacent Time Point Comparisons**

,up genes,down genes
comparison,,
gastrocnemius female female 1w v female sedentary,82,121
gastrocnemius female female 2w v female 1w,10,10
gastrocnemius female female 4w v female 2w,6,13
gastrocnemius female female 8w v female 4w,3,2


**Time Point Versus 0 Comparisons**

,up genes,down genes
comparison,,
gastrocnemius female female 1w v female sedentary,82,121
gastrocnemius female female 2w v female sedentary,24,29
gastrocnemius female female 4w v female sedentary,39,24
gastrocnemius female female 8w v female sedentary,17,39
